<a href="https://colab.research.google.com/github/michael-palomino-tm/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/blob/Mistral/RA3/IL3.1/3_1_3_Laboratorio_2_Trazabilidad_LangSmith_MercadoAndes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ISY0101 · Ingeniería de Soluciones con IA
## EA3 · Actividad 1 — Laboratorio 2: trazabilidad con LangSmith
**Caso de estudio:** MercadoAndes (empresa y datos ficticios).

En el Laboratorio 1 registramos eventos en un archivo JSONL. Ahora construiremos **trazas** para visualizar cómo una solicitud se divide en operaciones internas (**spans**), cuánto tarda cada operación y dónde se produjo un fallo.

**Al finalizar podrás:** identificar una traza raíz, reconocer sus operaciones hijas, encontrar un cuello de botella y distinguir un fallo de herramienta de un fallo del modelo.

## 1. Preparación del entorno
El notebook se ejecuta en **Google Colab**. Los cuatro parámetros de acceso se obtienen **únicamente desde Colab Secrets**: `LLM_API_KEY`, `LLM_MODEL_SMALL`, `LLM_BASE_URL` y `LANGSMITH_API_KEY`.

Activa el acceso de este notebook a los secretos en el panel **Secrets** de Colab. No pegues valores privados en el código ni los publiques en las capturas.

In [ ]:
# Instalamos las bibliotecas utilizadas en este laboratorio.
# - langchain-openai: cliente ChatOpenAI compatible con la API de Mistral.
# - langsmith: registro y exploración de trazas.
# - pandas: representación tabular de mediciones locales.
%pip -q install -U langchain-openai langsmith pandas

In [ ]:
import os
import time
import uuid
import pandas as pd
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langsmith import traceable

# Lee un secreto directamente del almacén seguro de Colab.
# No leemos archivos .env ni usamos os.getenv() como fuente de configuración.
def leer_secreto(nombre):
    try:
        valor = userdata.get(nombre)
    except Exception as exc:
        raise RuntimeError(f"No se pudo leer el secreto {nombre} en Colab.") from exc
    if not isinstance(valor, str) or not valor.strip():
        raise RuntimeError(f"Falta el secreto requerido: {nombre}")
    return valor.strip()

# Estos cuatro valores son secretos. Nunca los imprimimos.
LLM_API_KEY = leer_secreto("LLM_API_KEY")
LLM_MODEL_SMALL = leer_secreto("LLM_MODEL_SMALL")
LLM_BASE_URL = leer_secreto("LLM_BASE_URL")
LANGSMITH_API_KEY = leer_secreto("LANGSMITH_API_KEY")

# LangSmith y la instrumentación de LangChain leen estas variables.
# Aquí solo ESCRIBIMOS variables de entorno a partir de secretos de Colab:
# NO estamos leyendo configuración desde el entorno.
os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "ISY0101-EA3-Observabilidad"

# El nombre del modelo y su URL provienen exclusivamente de Colab Secrets.
llm = ChatOpenAI(
    model=LLM_MODEL_SMALL,
    api_key=LLM_API_KEY,
    base_url=LLM_BASE_URL,
    temperature=0,
    max_retries=1,  # Evita reintentos prolongados ante cuotas agotadas.
    timeout=30,
)
print("Configuración cargada. Proyecto LangSmith:", os.environ["LANGSMITH_PROJECT"])
print("Proveedor configurado; credenciales ocultas.")

### 1.1 ¿Dónde veremos las trazas?
Abre [LangSmith](https://smith.langchain.com/) en otra pestaña, ingresa al proyecto **ISY0101-EA3-Observabilidad** y localiza la sección de trazas o ejecuciones. El envío de trazas requiere acceso a Internet, una clave LangSmith válida y los permisos correspondientes.

Una **traza raíz** representa una solicitud completa. Una operación **hija** representa una parte de ese trabajo (por ejemplo, una herramienta o una llamada al LLM).

## 2. Datos y herramientas del caso
MercadoAndes es una empresa ficticia. Los datos son locales, controlados y no contienen información personal. La herramienta comercial retorna pedidos y reclamos; la de operaciones retorna demanda y capacidad de despachos.

Usaremos `@traceable` para indicar qué funciones deben aparecer como operaciones separadas en LangSmith.

In [ ]:
# Datos ficticios y constantes compartidas por todas las ejecuciones.
DATOS = {
    "comercial": {
        "pedidos_trimestre_anterior": 1200,
        "pedidos_trimestre_actual": 1020,
        "abandono_carritos_pct": 32,
        "reclamos_totales": 230,
        "reclamos_entrega_tardia": 138,
    },
    "operaciones": {
        "despachos_requeridos_semana": 400,
        "capacidad_despachos_semana": 320,
        "entregas_tardias_pct": 27,
    },
}

@traceable(run_type="tool", name="herramienta_datos_comerciales")
def consultar_comercial():
    # Un span de herramienta se crea cada vez que esta función es invocada.
    # Devolvemos una copia para no modificar accidentalmente los datos base.
    return dict(DATOS["comercial"])

@traceable(run_type="tool", name="herramienta_datos_operaciones")
def consultar_operaciones(lento=False):
    # Si lento=True, introducimos una pausa artificial y reproducible.
    # Es una DEMORA DIDÁCTICA, no una medición real de una API empresarial.
    if lento:
        time.sleep(1.5)
    return dict(DATOS["operaciones"])

@traceable(run_type="tool", name="herramienta_con_error")
def consultar_fuente_fallida():
    # Esta excepción se utiliza para demostrar cómo LangSmith registra un fallo.
    # No se consulta el modelo de lenguaje después de este error.
    raise RuntimeError("Fallo controlado: fuente de reclamos temporalmente no disponible")

print("Herramientas instrumentadas: comercial, operaciones y fallo controlado.")

## 3. Construcción del flujo trazable
El flujo hace tres cosas: consulta datos, solicita una síntesis a Mistral y devuelve una respuesta. La función principal se marca como **traza raíz**; dentro de ella se registrarán las llamadas a herramientas y al modelo.

Los tiempos de cada operación se almacenan automáticamente en LangSmith; además guardaremos algunas mediciones locales para analizarlas después.

In [ ]:
# La lista de resultados locales se usa solo para construir una tabla en Colab.
# LangSmith captura las trazas por separado en el proyecto configurado.
RESULTADOS = []

@traceable(run_type="chain", name="MercadoAndes_solicitud_completa")
def ejecutar_consulta(pregunta, escenario="normal"):
    # Cada solicitud tiene un identificador visible para buscarla en la plataforma.
    identificador = str(uuid.uuid4())[:8]
    inicio = time.perf_counter()  # Reloj apropiado para medir duración.
    try:
        # Paso 1: herramientas. Esta llamada aparece como span hijo.
        comercial = consultar_comercial()
        operaciones = consultar_operaciones(lento=(escenario == "lento"))

        # Paso 2: fallo reproducible, ejecutado solo si se solicita.
        # Al lanzar la excepción, la traza raíz también quedará marcada con error.
        if escenario == "fallido":
            consultar_fuente_fallida()

        # Paso 3: enviamos el contexto FICTICIO al modelo.
        # ChatOpenAI está instrumentado por LangChain y genera el span de LLM.
        mensajes = [
            ("system", "Eres un analista. Usa exclusivamente los datos entregados. "
                       "Distingue hechos de hipótesis; no inventes causalidad. "
                       "Responde en español en un máximo de 100 palabras."),
            ("human", f"Pregunta: {pregunta}\nDatos comerciales: {comercial}\n"
                      f"Datos operacionales: {operaciones}"),
        ]
        respuesta = llm.invoke(mensajes, config={"tags": ["ISY0101", escenario]})
        texto = respuesta.content
        estado = "ok"
        return {"id_local": identificador, "respuesta": texto, "escenario": escenario}
    except Exception as exc:
        # No mostramos el mensaje completo del error del proveedor: puede incluir
        # información técnica o sensible. La clase sí aparecerá en la consola.
        estado = "error"
        print(f"Error en {escenario}: {type(exc).__name__}")
        return {"id_local": identificador, "error_tipo": type(exc).__name__,
                "escenario": escenario}
    finally:
        # Siempre registramos duración y estado, incluso si hubo una excepción.
        # OJO: devolver un error atrapado evita que la traza raíz marque excepción;
        # el span de herramienta fallida sí conservará su fallo.
        duracion_ms = round((time.perf_counter() - inicio) * 1000, 2)
        RESULTADOS.append({"id_local": identificador, "escenario": escenario,
                           "estado": estado, "duracion_ms": duracion_ms})

### Interpretación de la estructura
En LangSmith deberías poder observar una jerarquía similar a esta:

`MercadoAndes_solicitud_completa` (raíz)  
→ `herramienta_datos_comerciales`  
→ `herramienta_datos_operaciones`  
→ `ChatOpenAI` / llamada al modelo

La herramienta de error aparecerá únicamente cuando ejecutemos el escenario fallido. El orden y nombres concretos de algunas operaciones internas pueden variar según la versión de las bibliotecas.

## 4. Ejecutar escenarios y comparar trazas
Ejecuta primero la solicitud normal, después la lenta y, por último, una ejecución con error deliberado. Busca cada traza en LangSmith y compara las duraciones de sus operaciones.

In [ ]:
# Ejecución normal: se consulta información y el LLM redacta la respuesta.
resultado_normal = ejecutar_consulta(
    "¿Qué señales sugieren problemas en ventas y logística?", escenario="normal"
)
print("Escenario normal:", resultado_normal)

# Ejecución lenta: exactamente el mismo objetivo, pero con una herramienta más lenta.
resultado_lento = ejecutar_consulta(
    "¿Qué señales sugieren problemas en ventas y logística?", escenario="lento"
)
print("Escenario lento:", resultado_lento)

# Ejecución fallida: produce un error en herramienta antes de invocar al LLM.
resultado_fallido = ejecutar_consulta(
    "Revisa las fuentes de la empresa.", escenario="fallido"
)
print("Escenario fallido:", resultado_fallido)

### 4.1 Explorar las trazas en LangSmith
En el proyecto **ISY0101-EA3-Observabilidad**, abre una ejecución de cada escenario y examina:

1. La traza raíz y sus operaciones hijas.
2. El span de herramienta que más demoró.
3. La llamada al modelo, su entrada y salida (solo datos ficticios).
4. El error controlado de la herramienta, si aparece en el escenario fallido.
5. Las métricas de tokens, **solo si el proveedor las entrega**.

**Importante:** una traza registra operaciones y datos instrumentados; no ofrece necesariamente acceso al razonamiento privado del modelo.

In [ ]:
# Tabla LOCAL de tiempos globales, para contrastar con lo mostrado en LangSmith.
# No reemplaza el análisis de spans: dos ejecuciones con igual duración total
# podrían tener cuellos de botella en componentes diferentes.
resumen = pd.DataFrame(RESULTADOS)
display(resumen)

# Como las llamadas reales dependen de la red y el proveedor, sus duraciones
# varían entre ejecuciones. Solo el retraso didáctico es deliberado.
if not resumen.empty:
    display(resumen.groupby("escenario", as_index=False).agg(
        cantidad=("estado", "size"),
        duracion_total_ms=("duracion_ms", "sum"),
    ))

## 5. Preguntas de análisis
Responde apoyándote en una **traza específica**, no solo en una impresión general.

1. ¿Cuál es la diferencia entre una traza raíz y un span hijo? Identifica ambos en tu ejecución.
2. ¿Qué herramienta aparece como cuello de botella en el escenario lento? ¿Qué evidencia lo demuestra?
3. ¿Por qué un registro de `request_end` o una respuesta visible no bastan para explicar dónde se consumió el tiempo?
4. En el escenario fallido, ¿qué operación registra el error? ¿Se ejecutó posteriormente el LLM?
5. ¿Qué diferencia observas entre duración total, duración de la herramienta y duración de la llamada al modelo?
6. ¿Qué datos no deberíamos registrar en una plataforma de trazas al trabajar con usuarios reales?
7. ¿Podemos afirmar que una respuesta es correcta solamente porque terminó sin errores? Fundamenta.

## 6. Evidencia y cierre
Como evidencia de esta actividad, conserva: una captura de una traza completa (sin secretos), otra de un span lento o fallido y una conclusión de tres o cuatro líneas con el cuello de botella identificado.

En el siguiente laboratorio utilizaremos estas ideas para construir indicadores y **comparar una solución antes y después de una mejora**.

**Referencias:** [LangSmith](https://docs.langchain.com/langsmith/observability), [LangChain y tracing](https://docs.langchain.com/langsmith/trace-with-langchain).